In [0]:
# 01_raw_dallas
# Reads Dallas CSV from volume
# Writes directly to Raw Delta table
# Logs to execution_log

dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("raw_schema",      "raw_zone")
dbutils.widgets.text("volume_name",     "volume_raw")
dbutils.widgets.text("metadata_schema", "metadata")
dbutils.widgets.text("file_name",       "Dallas.csv")

catalog_name    = dbutils.widgets.get("catalog_name")
raw_schema      = dbutils.widgets.get("raw_schema")
volume_name     = dbutils.widgets.get("volume_name")
metadata_schema = dbutils.widgets.get("metadata_schema")
file_name       = dbutils.widgets.get("file_name")

source_path = f"/Volumes/{catalog_name}/{raw_schema}/{volume_name}/{file_name}"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {raw_schema}")

print(f"Ingesting: {file_name}")
print(f"Source path: {source_path}")

In [0]:
# STEP 1: Reading Dallas CSV from volume

from datetime import datetime
from pyspark.sql.functions import current_timestamp

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(source_path)

df = df.withColumn("_raw_ingestion_timestamp", current_timestamp())

source_count = df.count()
print(f"Read {source_count} rows from {file_name}")
print(f"Columns: {len(df.columns)}")

In [0]:
# STEP 2: Clean column names
# Delta does not allow spaces or special chars

cleaned_cols = [
    col.strip()
       .replace(' ', '_')
       .replace('-', '_')
       .replace('*', '')
       .replace('/', '_')
       .replace('(', '')
       .replace(')', '')
       .replace('#', 'num')
    for col in df.columns
]
df = df.toDF(*cleaned_cols)

print(f"Column names cleaned")
print(f"Columns: {df.columns}")

In [0]:
# STEP 3: Write to Raw Delta table
# Exact copy of source. no transformations

raw_table = f"{catalog_name}.{raw_schema}.dallas"

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(raw_table)

target_count = spark.read.table(raw_table).count()
print(f"Written {target_count} rows to {raw_table}")

In [0]:
# STEP 4: Validate row counts

if source_count == target_count:
    status = "SUCCESS"
    print(f"Row count validated: {source_count} rows match")
else:
    status = "FAILED"
    print(f"Mismatch — source: {source_count} | target: {target_count}")

In [0]:
# STEP 5: Log to execution_log
from pyspark.sql import Row

log_row = spark.createDataFrame([Row(
    table_name       = "dallas",
    city             = "Dallas",
    execution_time   = datetime.now(),
    status           = status,
    source_row_count = source_count,
    target_row_count = target_count,
    file_location    = source_path,
    created_date     = datetime.now().date()
)])

log_row.write.format("delta") \
    .mode("append") \
    .saveAsTable(f"{catalog_name}.{metadata_schema}.execution_log")

print(f"Logged to execution_log. Status: {status}")

In [0]:
# STEP 6: Verify

print("Raw Dallas table preview:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{raw_schema}.dallas LIMIT 5"))

In [0]:
print("Execution log:")
display(spark.sql(f"SELECT * FROM {catalog_name}.{metadata_schema}.execution_log"))

In [0]:
# STEP 7: Fail task if validation failed

if status == "FAILED":
    raise Exception(f"Row count mismatch for Dallas. Check execution_log.")